In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!cp "/content/drive/MyDrive/Year 3/IML/ex4/whichfaceisreal.zip" .
!unzip -q whichfaceisreal.zip
print("Done! Files in current directory:")
!ls

In [ ]:
import os
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights
import torch
import torchvision
from tqdm import tqdm
from torchvision import transforms
import numpy as np
from xgboost import XGBClassifier

def get_loaders(path, transform, batch_size):
    """
    Get the data loaders for the train, validation and test sets.
    :param path: The path to the 'whichfaceisreal' directory.
    :param transform: The transform to apply to the images.
    :param batch_size: The batch size.
    :return: The train, validation and test data loaders.
    """
    train_set = torchvision.datasets.ImageFolder(root=os.path.join(path, 'train'), transform=transform)
    val_set = torchvision.datasets.ImageFolder(root=os.path.join(path, 'val'), transform=transform)
    test_set = torchvision.datasets.ImageFolder(root=os.path.join(path, 'test'), transform=transform)

    train_loader = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_loader = torch.utils.data.DataLoader(val_set, batch_size=batch_size, shuffle=False)
    test_loader = torch.utils.data.DataLoader(test_set, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader

# Set the random seed for reproducibility
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
transform = transforms.Compose([transforms.Resize((224, 224)), transforms.CenterCrop(64), transforms.ToTensor()])
batch_size = 32
path = './whichfaceisreal'
train_loader, val_loader, test_loader = get_loaders(path, transform, batch_size)


# DATA LOADING
### DO NOT CHANGE THE CODE BELOW THIS LINE ###
train_data = []
train_labels = []
test_data = []
test_labels = []
with torch.no_grad():
    for (imgs, labels) in tqdm(train_loader, total=len(train_loader), desc='Train'):
        train_data.append(imgs)
        train_labels.append(labels)
    train_data = torch.cat(train_data, 0).cpu().numpy().reshape(len(train_loader.dataset), -1)
    train_labels = torch.cat(train_labels, 0).cpu().numpy()
    for (imgs, labels) in tqdm(test_loader, total=len(test_loader), desc='Test'):
        test_data.append(imgs)
        test_labels.append(labels)
    test_data = torch.cat(test_data, 0).cpu().numpy().reshape(len(test_loader.dataset), -1)
    test_labels = torch.cat(test_labels, 0).cpu().numpy()
### DO NOT CHANGE THE CODE ABOVE THIS LINE ###


### YOUR XGBOOST CODE GOES HERE ###
xgb_model = XGBClassifier(random_state=0)
xgb_model.fit(train_data, train_labels) #start
train_accuracy = xgb_model.score(train_data, train_labels)
test_accuracy = xgb_model.score(test_data, test_labels)

print(f"\nXGBoost Results:")
print(f"Train Accuracy: {train_accuracy:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
#save predictions
xgb_preds = xgb_model.predict(test_data)


In [16]:
import os
import torch
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights
import torch
import torchvision
from tqdm import tqdm
from torchvision import transforms
import numpy as np

class ResNet18(nn.Module):
    def __init__(self, pretrained=False, probing=False):
        super(ResNet18, self).__init__()
        if pretrained:
            weights = ResNet18_Weights.IMAGENET1K_V1
            self.transform = ResNet18_Weights.IMAGENET1K_V1.transforms()
            self.resnet18 = resnet18(weights=weights)
        else:
            self.transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])
            self.resnet18 = resnet18()

        in_features_dim = self.resnet18.fc.in_features
        self.resnet18.fc = nn.Identity()
        if probing:
            for name, param in self.resnet18.named_parameters():
                param.requires_grad = False
        self.logistic_regression = nn.Linear(in_features_dim, 1)

    def forward(self, x):
        features = self.resnet18(x)
        ### YOUR CODE HERE ###
        return self.logistic_regression(features) #features from resnet18 and run on layer

def get_loaders(path, transform, batch_size):
    """
    Get the data loaders for the train, validation and test sets.
    """
    train_set = torchvision.datasets.ImageFolder(root=os.path.join(path, 'train'), transform=transform)
    val_set = torchvision.datasets.ImageFolder(root=os.path.join(path, 'val'), transform=transform)
    test_set = torchvision.datasets.ImageFolder(root=os.path.join(path, 'test'), transform=transform)

    train_loader = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_loader = torch.utils.data.DataLoader(val_set, batch_size=batch_size, shuffle=False)
    test_loader = torch.utils.data.DataLoader(test_set, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader

def compute_accuracy(model, data_loader, device):
    """
    Compute the accuracy of the model on the data in data_loader
    """
    model.eval()
    ### YOUR CODE HERE ###
    correct = 0
    total = 0
    with torch.no_grad():
        for imgs, labels in data_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            #output >0 mean class 1 else class 0
            predictions = (outputs > 0).float().view(-1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
    return correct / total if total > 0 else 0

def run_training_epoch(model, criterion, optimizer, train_loader, device):
    """
    Run a single training epoch
    :param model: The model to train
    :param criterion: The loss function
    :param optimizer: The optimizer
    :param train_loader: The data loader
    :param device: The device to run the training on
    :return: The average loss for the epoch.
    """
    model.train()
    total_loss = 0
    for (imgs, labels) in tqdm(train_loader, total=len(train_loader)):
        ### YOUR CODE HERE ###
        imgs, labels = imgs.to(device), labels.to(device).float().view(-1, 1)

        optimizer.zero_grad()
        outputs = model(imgs) #forward
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    return total_loss / len(train_loader)

In [ ]:
#training from scratch with different learning rates
path = './whichfaceisreal'
learning_rates = [1e-1, 1e-2, 1e-3, 1e-4, 1e-5]
scratch_results = {}

for lr in learning_rates:
    print(f"\nEvaluating Scratch with LR: {lr}")
    torch.manual_seed(42)

    #build model
    model = ResNet18(pretrained=False, probing=False)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    #loaders for this model
    train_loader, val_loader, test_loader = get_loaders(path, model.transform, batch_size=32)
    criterion = torch.nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    #run one epoch of training
    loss = run_training_epoch(model, criterion, optimizer, train_loader, device)

    #check test accuracy
    test_accuracy = compute_accuracy(model, test_loader, device)
    print(f"LR {lr} - Loss: {loss:.4f}, Test Accuracy: {test_accuracy:.4f}")
    scratch_results[lr] = test_accuracy



In [ ]:
#linerar probing
path = './whichfaceisreal'
learning_rates = [1e-1, 1e-2, 1e-3, 1e-4, 1e-5]
probing_results = {}

for lr in learning_rates:
    print(f"\nEvaluating Linear Probing with LR: {lr}")
    torch.manual_seed(42)

    #create model with probing=True
    model = ResNet18(pretrained=True, probing=True)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    #loading data: using model.transform (ImageNet transformation)
    train_loader, val_loader, test_loader = get_loaders(path, model.transform, batch_size=32)
    criterion = torch.nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr) #to update last layer only

    loss = run_training_epoch(model, criterion, optimizer, train_loader, device)
    test_accuracy = compute_accuracy(model, test_loader, device)
    print(f"LR {lr} - Loss: {loss:.4f}, Test Accuracy: {test_accuracy:.4f}")
    probing_results[lr] = test_accuracy #save results



In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression

#sklearn-based linear probing
torch.manual_seed(42)
np.random.seed(42)

model = ResNet18(pretrained=True, probing=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
model.eval()

#turn images to features
def extract_features(loader):
    features_list = []
    labels_list = []

    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc="Extracting"):
            imgs = imgs.to(device)
            #get features from model
            features = model.resnet18(imgs)
            features_list.append(features.cpu().numpy())
            labels_list.append(labels.numpy())

    return np.concatenate(features_list), np.concatenate(labels_list)

#get data loaders
train_loader, val_loader, test_loader = get_loaders(path, model.transform, batch_size=32)
X_train, y_train = extract_features(train_loader)
X_test, y_test = extract_features(test_loader)

#train sklearn logistic regression
print(f"Training LogisticRegression on features of shape: {X_train.shape}...")
sklearn_clf = LogisticRegression(max_iter=1000) #max_iter 1000 to has time finish training
sklearn_clf.fit(X_train, y_train)

#evaluate
train_accuracy = sklearn_clf.score(X_train, y_train)
test_accuracy = sklearn_clf.score(X_test, y_test)

print(f"Train Accuracy: {train_accuracy:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

In [ ]:
#fine tuning evaluation
path = './whichfaceisreal'
learning_rates = [1e-1, 1e-2, 1e-3, 1e-4, 1e-5]
finetuning_results = {}

for lr in learning_rates:
    print(f"\nEvaluating Fine-tuning with LR: {lr}")
    torch.manual_seed(42)

    #start with image net weights
    model = ResNet18(pretrained=True, probing=False)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    train_loader, val_loader, test_loader = get_loaders(path, model.transform, batch_size=32)
    criterion = torch.nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    loss = run_training_epoch(model, criterion, optimizer, train_loader, device)
    test_acc = compute_accuracy(model, test_loader, device)
    print(f"LR {lr} - Loss: {loss:.4f}, Test Accuracy: {test_acc:.4f}")
    finetuning_results[lr] = test_acc



In [ ]:
import matplotlib.pyplot as plt
import torch

# Visualize 5 samples
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def get_detailed_predictions(model, loader):
    model.eval()
    all_images = []
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            preds = (outputs > 0).float().view(-1) #0.5 threshold for binary decision

            #for plotting later
            all_images.append(imgs.cpu())
            all_labels.append(labels.cpu())
            all_preds.append(preds.cpu())

    return torch.cat(all_images), torch.cat(all_labels), torch.cat(all_preds)

#get best model and train little
best_model = ResNet18(pretrained=True, probing=False).to(device)
train_loader, val_loader, test_loader = get_loaders(path, best_model.transform, batch_size=32)
optimizer_best = torch.optim.Adam(best_model.parameters(), lr=0.0001)
run_training_epoch(best_model, torch.nn.BCEWithLogitsLoss(), optimizer_best, train_loader, device)

#worst model
worst_model = ResNet18(pretrained=False, probing=False).to(device)
optimizer_worst = torch.optim.Adam(worst_model.parameters(), lr=0.0001)
run_training_epoch(worst_model, torch.nn.BCEWithLogitsLoss(), optimizer_worst, train_loader, device)

#predict on test set
test_imgs, test_labels, best_preds = get_detailed_predictions(best_model, test_loader)
_, _, worst_preds = get_detailed_predictions(worst_model, test_loader)
#find where best is correct and worst is wrong
correct_mask = (best_preds == test_labels) & (worst_preds != test_labels)
indices = torch.where(correct_mask)[0]
#show 5 samples
plt.figure(figsize=(20, 4))
for i in range(min(5, len(indices))):
    idx = indices[i]
    img = test_imgs[idx].permute(1, 2, 0).numpy()

    #normalize for display
    img = (img - img.min()) / (img.max() - img.min())
    plt.subplot(1, 5, i + 1)
    plt.imshow(img)
    label_name = "Real" if test_labels[idx] == 1 else "Fake"
    plt.title(f"Sample {i+1}\nTrue Label: {label_name}")
    plt.axis('off')
plt.tight_layout()
plt.show()

if len(indices) < 5:
    print(f"Note: Only found {len(indices)} samples that match the criteria.")